# Gemma 3 4B IT: harmonized inference robustness v1

This notebook executes the frozen Stage-2 robustness experiment under
NF4/BF16, SDPA, one-shot, no-cache inference with sequential causal
visibility. It first runs an outcome-independent engineering preflight
on excluded seed 8999. Only then can it write the 256 confirmatory rows
for fresh seeds 9000--9015.

Results are append-only and resume-safe. Analysis remains locked until
all rows, four-cell pairing invariants, the engineering-audit hash, and
the stable runtime fingerprint pass. All outcomes are reported.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, shutil, subprocess, sys

PROJECT_DIR = Path('/content/drive/MyDrive/tacl_factorization_stage2')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
HARNESS_ZIP = PROJECT_DIR / 'TACL_Expansion_Harness_v5_1.zip'
REPO_DIR = Path('/content/causal_factorization')

if not HARNESS_ZIP.exists():
    print('Upload TACL_Expansion_Harness_v5_1.zip once; it will remain in Drive.')
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith('.zip')]
    if not candidates:
        raise RuntimeError('No harness ZIP was uploaded.')
    shutil.copy2(candidates[0], HARNESS_ZIP)

if not REPO_DIR.exists():
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/bilkentli88/causal_factorization', str(REPO_DIR)
    ])

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.15.1', 'accelerate', 'bitsandbytes', 'tokenizers'
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    str(HARNESS_ZIP), str(REPO_DIR / 'code/fragburden0.3.0.zip')
])
for module_name in list(sys.modules):
    if module_name == 'tacl_factorization' or module_name.startswith('tacl_factorization.'):
        del sys.modules[module_name]
import importlib
importlib.invalidate_caches()
print('Setup complete.')

In [ ]:
from tacl_factorization.config import assert_frozen, config_hash, load_config
import tacl_factorization

assert tacl_factorization.__version__ == '0.5.1'
EXTRACTED = Path('/content/tacl_harness_stage2')
if EXTRACTED.exists():
    shutil.rmtree(EXTRACTED)
shutil.unpack_archive(str(HARNESS_ZIP), str(EXTRACTED))
matches = list(EXTRACTED.rglob('gemma_harmonized_robustness_v1.json'))
if len(matches) != 1:
    raise RuntimeError('Could not locate the frozen Stage-2 configuration.')
cfg = load_config(matches[0])
assert_frozen(cfg)
cfg_hash = config_hash(cfg)
assert cfg['expected_rows'] == 256
assert cfg['semantic_seeds'] == list(range(9000, 9016))
print('Frozen config hash:', cfg_hash)
print(json.dumps(cfg, indent=2))

In [ ]:
import inspect, platform
import torch, transformers, bitsandbytes
from google.colab import userdata
from huggingface_hub import HfApi
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

assert torch.cuda.is_available(), 'Select a GPU runtime.'
assert torch.cuda.is_bf16_supported(), 'Stage 2 requires a BF16-capable GPU.'
assert transformers.__version__ == cfg['transformers_version']
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if not hf_token:
    raise RuntimeError('Add an HF_TOKEN Colab secret with access to gated Gemma weights.')

resolved_revision = HfApi().model_info(
    cfg['model'], revision=cfg['requested_revision'], token=hf_token
).sha
if not resolved_revision:
    raise RuntimeError('Could not resolve the gated model revision.')
tokenizer = AutoTokenizer.from_pretrained(
    cfg['model'], revision=resolved_revision, use_fast=True, token=hf_token
)
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
load_kwargs = dict(
    revision=resolved_revision, quantization_config=bnb, device_map='auto',
    attn_implementation='sdpa', token=hf_token
)
try:
    model = Gemma3ForConditionalGeneration.from_pretrained(
        cfg['model'], dtype=torch.bfloat16, **load_kwargs
    ).eval()
except TypeError:
    model = Gemma3ForConditionalGeneration.from_pretrained(
        cfg['model'], torch_dtype=torch.bfloat16, **load_kwargs
    ).eval()
if hasattr(model.config, 'use_cache'):
    model.config.use_cache = False

runtime = {
    'model': cfg['model'],
    'resolved_model_revision': resolved_revision,
    'resolved_tokenizer_revision': resolved_revision,
    'model_class': type(model).__name__,
    'weights': cfg['weights'],
    'compute_dtype': cfg['compute_dtype'],
    'attention_implementation': cfg['attention_implementation'],
    'inference_path': cfg['inference_path'],
    'scorer_version': cfg['scorer_version'],
    'harness_version': tacl_factorization.__version__,
    'transformers': transformers.__version__,
    'torch': torch.__version__,
    'bitsandbytes': bitsandbytes.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'cuda_capability': list(torch.cuda.get_device_capability(0)),
}
runtime_fingerprint = hashlib.sha256(
    json.dumps(runtime, sort_keys=True, separators=(',', ':')).encode()
).hexdigest()[:24]
print(json.dumps(runtime, indent=2))
print('Stable runtime fingerprint:', runtime_fingerprint)

In [ ]:
from collections import Counter
from fragburden.dial import FragmentationDial, verify_monotone
from fragburden.testing import sample_texts
from tacl_factorization.adapters import HFChatAdapter
from tacl_factorization.qualification import static_model_checks

static_gate = static_model_checks(
    model, tokenizer, required_context=max(cfg['target_lengths']),
    candidate_token_ids=cfg['candidate_token_ids']
)
assert static_gate['passed'], static_gate
assert type(model).__name__ == cfg['model_class']
text_config = model.config.text_config
layer_types = getattr(text_config, 'layer_types', None)
if layer_types is None:
    pattern = int(getattr(text_config, 'sliding_window_pattern', 6))
    layer_types = [
        'full_attention' if ((index + 1) % pattern == 0) else 'sliding_attention'
        for index in range(int(text_config.num_hidden_layers))
    ]
layer_counts = {key: int(value) for key, value in Counter(layer_types).items()}
assert layer_counts == cfg['expected_layer_type_counts'], layer_counts
assert int(text_config.sliding_window) == cfg['expected_sliding_window']

dial = FragmentationDial.from_transformers(tokenizer, name=cfg['model'])
dial_gate = verify_monotone(
    dial, sample_texts('en') + sample_texts('tr'),
    levels=(100, *sorted(set(cfg['frag_keep_by_language'].values()), reverse=True))
)
assert dial_gate['monotone_nondecreasing'] and dial_gate['all_in_vocab']
adapter = HFChatAdapter(
    tokenizer=tokenizer,
    base_dial=dial.at(100),
    fragmented_dials={
        language: dial.at(cfg['frag_keep_by_language'][language])
        for language in cfg['filler_languages']
    },
    chat_kwargs=cfg['chat_kwargs'],
    candidate_token_ids=cfg['candidate_token_ids'],
    shell_extraction=cfg['chat_shell_extraction'],
    encoding_strategy=cfg['region_labeling'],
    marker='UNIQUE_USER_CONTENT_MARKER_GEMMA_V070_A93F27',
)
assert len(adapter.prefix_ids) == cfg['expected_chat_prefix_tokens']
assert len(adapter.official_suffix_ids) == cfg['expected_chat_suffix_tokens']
assert adapter.digit_ids == cfg['candidate_token_ids']

probe_user = (
    'Task: look up the value for the queried key.\nEVIDENCE:\n'
    'K10000=1\nK10001=2\nK10002=3\nK10003=4\n'
    'QUERY KEY: K10000\nAnswer with one digit only:'
)
prompt_ids = adapter.apply_chat_ids(probe_user)
for digit in '1234':
    complete = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': probe_user}, {'role': 'assistant', 'content': digit}],
        tokenize=True, add_generation_prompt=False
    )
    if hasattr(complete, 'keys') and 'input_ids' in complete:
        complete = complete['input_ids']
    if hasattr(complete, 'detach'):
        complete = complete.detach().cpu().tolist()
    if complete and isinstance(complete[0], (list, tuple)):
        complete = complete[0]
    complete = [int(value) for value in complete]
    assert complete[:len(prompt_ids)] == prompt_ids
    continuation = [int(value) for value in complete[len(prompt_ids):]]
    assert continuation and continuation[0] == cfg['candidate_token_ids'][digit]
print('Architecture, tokenizer, fragmentation, chat-shell, and readout gates passed.')

## Engineering preflight

This excluded-seed stage may reject a broken implementation but cannot
choose a scientific outcome. It stores no predictions or logits.

In [ ]:
AUDIT_PATH = PROJECT_DIR / 'gemma_harmonized_engineering_audit_v1.json'
if AUDIT_PATH.exists():
    audit = json.loads(AUDIT_PATH.read_text())
    if (audit.get('config_hash') != cfg_hash or
            audit.get('runtime_fingerprint') != runtime_fingerprint or
            audit.get('passed') is not True):
        raise RuntimeError(
            'Existing engineering audit belongs to another config/runtime or did not pass. '
            'Use a separately named Stage-2 project directory; do not overwrite provenance.'
        )
    audit_sha = hashlib.sha256(AUDIT_PATH.read_bytes()).hexdigest()
    print('PASS: exact existing engineering audit reused:', audit_sha)
else:
    from tacl_factorization.generator import build_instance
    from tacl_factorization.geometry import (
        Condition, assemble_factorial, gap_diagnostics, hash_positions
    )
    from tacl_factorization.scoring import score_one_shot, score_gemma3_one_shot_sequential

    explicit_scorer = score_gemma3_one_shot_sequential

    def build_probe(language, target_tokens, trial_index=0):
        instance = build_instance(
            filler_language=language,
            semantic_seed=cfg['engineering_seed'],
            trial_index=trial_index,
            target_chat_tokens=target_tokens,
            count_chat_tokens=adapter.count_chat_tokens,
            verify_chat_shell=adapter.verify_chat_shell,
            n_records=cfg['n_records'],
        )
        encoding = adapter.encode_instance(instance)
        if adapter.last_encoding_strategy != cfg['region_labeling']:
            raise RuntimeError(
                f'Encoding strategy drift: {adapter.last_encoding_strategy} '
                f'!= {cfg["region_labeling"]}'
            )
        factorial = assemble_factorial(adapter.prefix_ids, adapter.suffix_ids, encoding)
        return instance, factorial

    probe_instance, probe = build_probe('en', 4096)
    assert probe['CC'].ids_hash == probe['CE'].ids_hash
    assert probe['FC'].ids_hash == probe['FE'].ids_hash
    assert abs(probe['CC'].gaps['evidence_readout'] - probe['FC'].gaps['evidence_readout']) <= 1e-3
    assert abs(probe['CE'].gaps['evidence_readout'] - probe['FE'].gaps['evidence_readout']) <= 1e-3

    native = score_one_shot(
        model, probe['CC'], probe_instance.gold_digit, adapter.digit_ids, tokenizer
    )
    explicit_1 = explicit_scorer(
        model, probe['CC'], probe_instance.gold_digit, adapter.digit_ids, tokenizer
    )
    explicit_2 = explicit_scorer(
        model, probe['CC'], probe_instance.gold_digit, adapter.digit_ids, tokenizer
    )
    native_argmax_equal = native['predicted_digit'] == explicit_1['predicted_digit']
    repeat_max_diff = max(
        abs(explicit_1['candidate_logits'][digit] - explicit_2['candidate_logits'][digit])
        for digit in '1234'
    )

    shifted_positions = probe['CC'].positions.copy()
    shifted_positions[-1] += 0.5
    shifted = Condition(
        code='PX', name='engineering_fractional_readout_shift',
        ids=probe['CC'].ids, positions=shifted_positions, labels=probe['CC'].labels,
        ids_hash=probe['CC'].ids_hash,
        position_hash=hash_positions(shifted_positions),
        gaps=gap_diagnostics(shifted_positions, len(adapter.prefix_ids), probe['CC'].labels),
    )
    fractional = explicit_scorer(
        model, shifted, probe_instance.gold_digit, adapter.digit_ids, tokenizer
    )
    fractional_max_diff = max(
        abs(explicit_1['candidate_logits'][digit] - fractional['candidate_logits'][digit])
        for digit in '1234'
    )

    hard_instance, hard_probe = build_probe('en', 8192, trial_index=1)
    hard_score = explicit_scorer(
        model, hard_probe['FE'], hard_instance.gold_digit, adapter.digit_ids, tokenizer
    )
    hard_finite = all(
        __import__('math').isfinite(hard_score['candidate_logits'][digit]) for digit in '1234'
    )

    gates = {
        'static_model': bool(static_gate['passed']),
        'fragmentation_monotone_and_in_vocab': bool(
            dial_gate['monotone_nondecreasing'] and dial_gate['all_in_vocab']
        ),
        'chat_shell_exact': True,
        'factorial_geometry_invariants': True,
        'native_vs_explicit_candidate_argmax_equal': bool(native_argmax_equal),
        'deterministic_repeat': repeat_max_diff == 0.0,
        'fractional_position_sensitivity': fractional_max_diff > 0.0,
        'hardest_fe_8k_finite': bool(hard_finite),
    }
    if 'model_architecture_exact' in cfg['engineering_gates']:
        gates['model_architecture_exact'] = True
    assert all(gates.values()), gates

    audit = {
        'protocol_version': cfg['protocol_version'],
        'config_hash': cfg_hash,
        'model': cfg['model'],
        'engineering_seed': cfg['engineering_seed'],
        'engineering_seed_excluded_from_analysis': True,
        'runtime': runtime,
        'runtime_fingerprint': runtime_fingerprint,
        'encoding_strategy': cfg['region_labeling'],
        'frag_keep_by_language': cfg['frag_keep_by_language'],
        'digit_ids': adapter.digit_ids,
        'static_gate': static_gate,
        'layer_type_counts': globals().get('layer_counts'),
        'gates': gates,
        'repeat_max_candidate_logit_difference': repeat_max_diff,
        'fractional_max_candidate_logit_difference': fractional_max_diff,
        'hardest_fe_8k_peak_cuda_gb': hard_score.get('peak_cuda_gb'),
        'task_predictions_or_logits_stored': False,
        'passed': True,
    }
    AUDIT_PATH.write_text(json.dumps(audit, indent=2, ensure_ascii=False) + '\n')
    audit_sha = hashlib.sha256(AUDIT_PATH.read_bytes()).hexdigest()
    print(json.dumps({
        'gates': gates,
        'repeat_max_candidate_logit_difference': repeat_max_diff,
        'fractional_max_candidate_logit_difference': fractional_max_diff,
        'hardest_fe_8k_peak_cuda_gb': hard_score.get('peak_cuda_gb'),
        'audit_sha256': audit_sha,
    }, indent=2))
    print('PASS: outcome-independent Stage-2 engineering preflight.')

## Frozen robustness run

Review the printed configuration and passed audit. Then change only
`RUN_HARMONIZED_ROBUSTNESS` to `True`. Interruptions are safe: rerun the
notebook under the same stable runtime and completed result IDs are skipped.

In [ ]:
from tacl_factorization.results import read_jsonl

RESULTS_PATH = PROJECT_DIR / 'gemma_harmonized_robustness_v1.jsonl'
existing = read_jsonl(RESULTS_PATH)
if existing:
    fingerprints = {row.get('runtime_fingerprint') for row in existing}
    config_hashes = {row.get('factorial_config_hash') for row in existing}
    audit_hashes = {row.get('engineering_audit_sha256') for row in existing}
    if fingerprints != {runtime_fingerprint} or config_hashes != {cfg_hash} or audit_hashes != {audit_sha}:
        raise RuntimeError(
            'Existing rows belong to another frozen config, audit, or stable runtime. '
            'Restore that environment or use a separately named result file; never mix rows.'
        )
print('Existing rows:', len(existing), '/', cfg['expected_rows'])

RUN_HARMONIZED_ROBUSTNESS = False
if not RUN_HARMONIZED_ROBUSTNESS:
    print('Run is armed but not started. Review the config and audit, then change only the flag to True.')

In [ ]:
if RUN_HARMONIZED_ROBUSTNESS:
    from tacl_factorization.experiment import run_symbolic_factorial
    from tacl_factorization.scoring import score_gemma3_one_shot_sequential

    provenance = {
        'factorial_config_hash': cfg_hash,
        'engineering_audit_sha256': audit_sha,
        'runtime_fingerprint': runtime_fingerprint,
        'runtime': runtime,
        'region_labeling': cfg['region_labeling'],
    }
    new_rows = run_symbolic_factorial(
        config=cfg, adapter=adapter, model=model,
        output_path=RESULTS_PATH, scorer=score_gemma3_one_shot_sequential,
        limit_new_rows=None, row_provenance=provenance,
    )
    print('New rows written:', new_rows)
    print('Result file:', RESULTS_PATH)

## Locked analysis

No estimate is calculated from a partial or structurally invalid run.
Direction agreement is descriptive and never filters the results.

In [ ]:
from tacl_factorization.statistics import (
    canonical_metrics, load_jsonl, summarize_factorial, validate_factorial_rows
)
import pandas as pd

rows = load_jsonl(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame()
print('Rows present:', len(rows), '/', cfg['expected_rows'])
if len(rows) != cfg['expected_rows']:
    print('INCOMPLETE: the analysis remains locked.')
else:
    validate_factorial_rows(rows)
    assert set(rows['runtime_fingerprint']) == {runtime_fingerprint}
    assert set(rows['factorial_config_hash']) == {cfg_hash}
    assert set(rows['engineering_audit_sha256']) == {audit_sha}
    base = ['filler_language', 'target_chat_tokens', 'semantic_seed', 'trial_index']
    assert (rows.groupby(base)['prompt_sha256'].nunique() == 1).all()
    pair = rows.pivot_table(
        index=base, columns='factorial_code',
        values=['ids_hash', 'evidence_readout_position_gap'], aggfunc='first'
    )
    assert (pair['ids_hash']['CC'] == pair['ids_hash']['CE']).all()
    assert (pair['ids_hash']['FC'] == pair['ids_hash']['FE']).all()
    assert ((pair['evidence_readout_position_gap']['CC'] - pair['evidence_readout_position_gap']['FC']).abs() <= 1e-3).all()
    assert ((pair['evidence_readout_position_gap']['CE'] - pair['evidence_readout_position_gap']['FE']).abs() <= 1e-3).all()
    assert (pair['evidence_readout_position_gap']['FE'] > pair['evidence_readout_position_gap']['FC']).all()
    print('PASS: completeness, pairing, geometry, prompt, audit, and runtime gates.')

    pooled = summarize_factorial(
        rows, bootstrap_seed=cfg['bootstrap']['seed'], reps=cfg['bootstrap']['reps']
    )
    cells = {}
    for (language, length), group in rows.groupby(['filler_language', 'target_chat_tokens']):
        cells[f'{language}_{int(length)}'] = summarize_factorial(
            group, bootstrap_seed=cfg['bootstrap']['seed'], reps=cfg['bootstrap']['reps']
        )

    position = pooled['position_effect_canonical_ids']['mean']
    representation = pooled['representation_effect_expanded_geometry']['mean']
    total = pooled['total_fragmentation_shift']['mean']
    denominator = abs(position) + abs(representation)
    cancellation_ratio = abs(total) / denominator if denominator else None
    cancellation_pass = (
        cancellation_ratio is not None
        and cancellation_ratio <= cfg['robustness_evaluation']['gemma_qualitative_cancellation_threshold']
    )

    observed_directions = {
        effect: ('positive' if values['mean'] > 0 else 'negative' if values['mean'] < 0 else 'zero')
        for effect, values in pooled.items()
    }
    direction_agreement = {
        effect: (
            None if reference == 'near_zero_cancellation'
            else observed_directions[effect] == reference
        )
        for effect, reference in cfg['historical_direction_reference'].items()
    }
    summary = {
        'protocol_version': cfg['protocol_version'],
        'factorial_config_hash': cfg_hash,
        'engineering_audit_sha256': audit_sha,
        'runtime_fingerprint': runtime_fingerprint,
        'rows': len(rows),
        'canonical_metrics': canonical_metrics(rows),
        'pooled_effects': pooled,
        'cell_effects': cells,
        'historical_direction_reference': cfg['historical_direction_reference'],
        'observed_directions': observed_directions,
        'direction_agreement': direction_agreement,
        'gemma_cancellation_ratio': cancellation_ratio,
        'gemma_qualitative_cancellation_pass': cancellation_pass,
        'reported_regardless_of_outcome': True,
    }
    SUMMARY_PATH = PROJECT_DIR / 'gemma_harmonized_robustness_v1_summary.json'
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + '\n')
    display(pd.DataFrame(pooled).T)
    print('Frozen summary:', SUMMARY_PATH)